In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance
np.set_printoptions(legacy='1.25')

In [ ]:
plt.rcParams["figure.figsize"] = [16, 9]
plt.rcParams["font.size"] = 20
plt.rcParams["axes.labelsize"] = 20
plt.rcParams["axes.titlesize"] = 24
plt.rcParams["xtick.labelsize"] = 16
plt.rcParams["ytick.labelsize"] = 16
plt.rcParams["font.family"] = "serif"

In [ ]:
stocks = ['KO', 'PEP', 'NVDA','KSU']
seq_len = 150

id = 'vb7m8jvt'
path_storage = f'storage/{id}'

seed, epoch = 42, 5
file_name = f'synthetics_epoch={epoch}_seed={seed}'

In [ ]:
real = pd.read_csv('data/stocks/midprice_volume__KO_PEP_NVDA_KSU__train.csv')[[f'mid_price_{s}' for s in stocks]].values
real = np.asarray([real[i*seq_len:(i+1)*seq_len] for i in range(len(real) // seq_len)]).transpose(0, 2, 1)
corrcoefs_real = np.asarray([np.corrcoef(s) for s in real])
n_samples = len(real)
real.shape, corrcoefs_real.shape

In [ ]:
with open(f'{path_storage}/inference_data/guidance_scale=0/{file_name}.npy', 'rb') as f:
    synthetic = np.load(f)
indices = np.random.choice(len(synthetic), n_samples, replace=False)
synthetic = synthetic[indices].transpose(0, 2, 1)
corrcoefs_synthetic_noGuidance = np.asarray([np.corrcoef(s) for s in synthetic])

In [ ]:
with open(f'{path_storage}/inference_data/guidance_scale=250/critic_epoch=121/{file_name}.npy', 'rb') as f:
    synthetic = np.load(f)
indices = np.random.choice(len(synthetic), n_samples, replace=False)
synthetic = synthetic[indices].transpose(0, 2, 1)
corrcoefs_synthetic_guidance = np.asarray([np.corrcoef(s) for s in synthetic])

In [ ]:
seed, epoch = 42, 8
file_name = f'synthetics_epoch={epoch}_seed={seed}'
with open(f'{path_storage}/inference_data_counterfactual/guidance_scale=500/critic_epoch=121/{file_name}.npy', 'rb') as f:
    synthetic = np.load(f)
indices = np.random.choice(len(synthetic), n_samples, replace=False)
synthetic = synthetic[indices].transpose(0, 2, 1)
corrcoefs_synthetic_guidance_counterfactual = np.asarray([np.corrcoef(s) for s in synthetic])

In [ ]:
colors = ['C1', 'Blue', '#017c90', '#642e19']
labels = ['Real', 'CSDI', 'CSDI + Critic', 'CSDI + Critic (CF)']
legend_elements = [
    plt.Rectangle((0,0), 1, 1, facecolor='white', edgecolor=c, label=l, linewidth=2) 
    for c, l in zip(colors, labels)
]
fig = plt.figure(figsize=(len(legend_elements), 1))
plt.legend(handles=legend_elements, loc='center', ncol=len(legend_elements))
plt.axis('off')
fig.savefig('legend.pdf', bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
I, J = np.triu_indices(len(stocks), k=1)

def make_plot(corrcoefs, c):    
    wass = dict()
    fig, axes = plt.subplots(1, 6, figsize=(16, 3))

    for i, j, ax in zip(I, J, axes):
        title = f'{stocks[i]} - {stocks[j]}'
        ax.set_title(title)

        hist = ax.hist(
            [corrcoefs[:, i, j], corrcoefs_real[:, i, j]], 
            color=[c, colors[0]],
            density=True, histtype="step", linewidth=2, 
        )
        _ = ax.hist(
            [corrcoefs[:, i, j], corrcoefs_real[:, i, j]], 
            color=[c, colors[0]],
            density=True, histtype="stepfilled", alpha=.3
        )
        wass[title] = wasserstein_distance(*hist[0]).round(2)

    axes[0].set_ylabel('Density')

    for ax in axes.ravel():
        ax.set_xlabel(r'$\rho$')
        ax.set_xlim((-1.1, 1.1))
        ax.set_ylim((0, 4))
        ax.set_facecolor('#eaeaf2')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.tick_params(bottom=False, left=False, which='both')
        ax.grid(True, linestyle='solid', c='w')

    fig.tight_layout()
    plt.savefig(f'correlations_{c}.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return wass

wass_noGuidance = make_plot(corrcoefs_synthetic_noGuidance, colors[1])
wass_guidance = make_plot(corrcoefs_synthetic_guidance, colors[2])
_ = make_plot(corrcoefs_synthetic_guidance_counterfactual, colors[3])

In [ ]:
wass = {
    'w=0': wass_guidance,
    'w=250': wass_noGuidance,
}
df = pd.DataFrame(wass)
df

In [ ]:
df.to_latex('correlations.tex', 
    float_format='%.2f',
    caption="Wasserstein distance - Critic as a guide.",
    label="tab:wasserstein_distance"
)